# Phase 3 — Validation & Baseline Modeling
### American Express Default Prediction

**Phase 1 + 2 recap (treated as source of truth):** `data/processed/train_features.parquet`
holds 458,913 customer rows, one per `customer_ID`, joined to `target` (25.89% default
rate), with 1,299 engineered features: ~1,225 continuous longitudinal aggregates
(mean/std/min/max/first/last/change over 175 continuous raw features), 39 categorical
longitudinal features (13 raw categorical variables × first/last/nunique), 33 selective
`_missing_rate` features, and 2 general history features. Phase 2 also flagged 8 raw
feature families as extremely sparse "candidates for dropping": `D_87`, `D_88`, `D_108`,
`D_110`, `D_111`, `B_39`, `D_73`, `B_42`.

**Goal of this notebook:** establish a trustworthy, reproducible baseline comparison of
three model families — Logistic Regression, LightGBM, CatBoost — on an identical
train/validation split, scored with the *actual* AMEX competition metric, not just ROC AUC.
No tuning, no ensembling, no test-set predictions.

**Same 8GB RAM constraint as Phases 1–2.** The processed dataset (2.6GB on disk) is small
enough to load once as a DataFrame — the discipline here is not creating *multiple* full
copies of it (dense one-hot matrices, redundant preprocessing outputs) simultaneously.


In [1]:
import os
import gc
import re
import time

import numpy as np
import pandas as pd
import psutil

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA_DIR = "../data"
PROCESSED_PATH = os.path.join(DATA_DIR, "processed", "train_features.parquet")
RANDOM_SEED = 42

_process = psutil.Process(os.getpid())
def rss_gb():
    return _process.memory_info().rss / 1e9

print(f"Starting RSS: {rss_gb():.2f} GB")


Starting RSS: 0.14 GB


### Package versions (reproducibility)


In [2]:
import sklearn, lightgbm, catboost, pyarrow
for name, mod in [("pandas", pd), ("numpy", np), ("scikit-learn", sklearn),
                   ("lightgbm", lightgbm), ("catboost", catboost), ("pyarrow", pyarrow)]:
    print(f"{name:<14} {mod.__version__}")


pandas         3.0.5
numpy          2.4.6
scikit-learn   1.9.0
lightgbm       4.7.0
catboost       1.2.10
pyarrow        22.0.0


## 1. Load and Validate the Processed Dataset

Loading the Parquet file directly — the raw 15GB `train_data.csv` is not touched in this
notebook.


In [3]:
t0 = time.time()
customer_features = pd.read_parquet(PROCESSED_PATH)
print(f"Loaded in {time.time()-t0:.1f}s. RSS: {rss_gb():.2f} GB")
print("Shape:", customer_features.shape)


Loaded in 4.7s. RSS: 1.43 GB
Shape: (458913, 1301)


In [4]:
checks = {}
checks["one row per customer_ID"] = customer_features["customer_ID"].is_unique
checks["row count == 458,913"] = customer_features.shape[0] == 458_913
checks["feature count == 1,299"] = customer_features.shape[1] - 2 == 1_299
checks["no missing target"] = customer_features["target"].isna().sum() == 0

numeric_cols_all = customer_features.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(customer_features[numeric_cols_all]).sum()
checks["no infinite values"] = inf_counts.sum() == 0

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
assert all(checks.values())

print()
print("Target distribution:")
print(customer_features["target"].value_counts(normalize=True).round(4))
print()
print("Dtype summary:")
print(customer_features.dtypes.value_counts())
print()
overall_missing_pct = customer_features.isna().mean().mean() * 100
print(f"Average missingness across all columns: {overall_missing_pct:.1f}%")


[PASS] one row per customer_ID
[PASS] row count == 458,913
[PASS] feature count == 1,299
[PASS] no missing target
[PASS] no infinite values

Target distribution:
target
0    0.7411
1    0.2589
Name: proportion, dtype: float64

Dtype summary:
float32    1258
float64      20
int64        16
str           5
int32         2
Name: count, dtype: int64



Average missingness across all columns: 13.7%


**Interpretation:** the file matches Phase 2's reported shape and target distribution
exactly, confirms the customer-level invariant (one row per customer) still holds, and has
zero infinite values. No recomputation of Phase 1/2 was needed — this is a direct check of
the artifact they produced.


## 8. Sparse-Feature Decision (applied here, before splitting)

Phase 2's own decision table flagged exactly **8 raw feature families** as
`"candidate for dropping later"` — not merely sparse, but sparse *and* with under 5%
customer coverage (`D_87`, `D_88`, `D_108`, `D_110`, `D_111`, `B_39`, `D_73`, `B_42`).
Per the preferred policy: **exclude the engineered columns derived from these 8 raw
variables from the modeling feature set, in this notebook only** — the saved Parquet file
itself is left untouched, so this decision is revisable without redoing Phase 2.

This is derived programmatically from the actual column names in the loaded file, not
retyped from the prompt.


In [5]:
DROP_RAW_FAMILIES = ["D_87", "D_88", "D_108", "D_110", "D_111", "B_39", "D_73", "B_42"]

drop_cols = [
    c for c in customer_features.columns
    if any(c == raw or c.startswith(raw + "_") for raw in DROP_RAW_FAMILIES)
]
print(f"Raw families excluded: {len(DROP_RAW_FAMILIES)}")
print(f"Engineered columns removed: {len(drop_cols)}")
for raw in DROP_RAW_FAMILIES:
    matches = [c for c in drop_cols if c == raw or c.startswith(raw + "_")]
    print(f"  {raw}: {matches}")

model_df = customer_features.drop(columns=drop_cols)
del customer_features
gc.collect()
print(f"\nmodel_df shape: {model_df.shape}. RSS: {rss_gb():.2f} GB")


Raw families excluded: 8
Engineered columns removed: 60
  D_87: ['D_87_first', 'D_87_last', 'D_87_nunique', 'D_87_missing_rate']
  D_88: ['D_88_mean', 'D_88_std', 'D_88_min', 'D_88_max', 'D_88_first', 'D_88_last', 'D_88_missing_rate', 'D_88_change']
  D_108: ['D_108_mean', 'D_108_std', 'D_108_min', 'D_108_max', 'D_108_first', 'D_108_last', 'D_108_missing_rate', 'D_108_change']
  D_110: ['D_110_mean', 'D_110_std', 'D_110_min', 'D_110_max', 'D_110_first', 'D_110_last', 'D_110_missing_rate', 'D_110_change']
  D_111: ['D_111_mean', 'D_111_std', 'D_111_min', 'D_111_max', 'D_111_first', 'D_111_last', 'D_111_missing_rate', 'D_111_change']
  B_39: ['B_39_mean', 'B_39_std', 'B_39_min', 'B_39_max', 'B_39_first', 'B_39_last', 'B_39_missing_rate', 'B_39_change']
  D_73: ['D_73_mean', 'D_73_std', 'D_73_min', 'D_73_max', 'D_73_first', 'D_73_last', 'D_73_missing_rate', 'D_73_change']
  B_42: ['B_42_mean', 'B_42_std', 'B_42_min', 'B_42_max', 'B_42_first', 'B_42_last', 'B_42_missing_rate', 'B_42_change


model_df shape: (458913, 1241). RSS: 1.32 GB


**No change of policy needed here** — the preferred default (exclude only the
"candidate for dropping" 8 families, keep everything else including the 25 sparse-but-some-
coverage features and their `_missing_rate` companions) is applied as given, since Phase 2
already did the harder analytical work of separating "almost never present" from
"often missing but present often enough to matter."


### Feature typing: numeric vs. categorical inputs

Phase 2 created three columns per categorical raw variable: `{col}_first`, `{col}_last`
(actual category labels) and `{col}_nunique` (a small integer *count* of distinct
categories seen). Only `_first`/`_last` are genuinely categorical for modeling purposes —
`_nunique` is a discrete numeric quantity and is treated as numeric input, not one-hot
encoded. Getting this distinction right matters for Section 5's Logistic Regression
preprocessing.


In [6]:
TRUE_CATEGORICAL_RAW = ["D_63", "D_64"]
CANDIDATE_CODE_RAW = ["D_116", "D_114", "D_66", "B_31", "D_120", "B_30", "D_126", "B_38", "D_117", "D_68"]  # D_87 excluded above
CATEGORICAL_RAW = TRUE_CATEGORICAL_RAW + CANDIDATE_CODE_RAW  # 12 (13 from Phase 2, minus dropped D_87)

categorical_cols = [f"{c}_first" for c in CATEGORICAL_RAW] + [f"{c}_last" for c in CATEGORICAL_RAW]
exclude_cols = ["customer_ID", "target"]
feature_cols = [c for c in model_df.columns if c not in exclude_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

assert set(numeric_cols) | set(categorical_cols) == set(feature_cols)
assert set(numeric_cols) & set(categorical_cols) == set()

print(f"Total modeling features: {len(feature_cols)}")
print(f"Numeric features (incl. {len(CATEGORICAL_RAW)} '_nunique' count columns): {len(numeric_cols)}")
print(f"Categorical features (first/last pairs, {len(CATEGORICAL_RAW)} raw variables): {len(categorical_cols)}")


Total modeling features: 1239
Numeric features (incl. 12 '_nunique' count columns): 1215
Categorical features (first/last pairs, 12 raw variables): 24


## 4. Leakage Check

Before training anything, three checks:


In [7]:
print("1. customer_ID in feature_cols:", "customer_ID" in feature_cols)
print("   target in feature_cols:     ", "target" in feature_cols)
assert "customer_ID" not in feature_cols and "target" not in feature_cols


1. customer_ID in feature_cols: False
   target in feature_cols:      False


In [8]:
# 2. Could any engineered feature have accidentally used target information?
# Every engineered column in feature_cols is a longitudinal aggregation (mean/std/min/max/
# first/last/change/nunique/missing_rate) computed in Phase 2 directly from raw statement
# rows (customer_ID, S_2, and the raw D_/S_/P_/B_/R_ columns) BEFORE target was ever joined
# in -- Phase 2's own pipeline joined target only at the very end, after all aggregation.
# So there is no code path by which target could have entered any feature's construction.
print("No feature construction step in Phase 2 used target -- target was joined only")
print("after all customer-level aggregation was complete (see Phase 2, Section 1).")


No feature construction step in Phase 2 used target -- target was joined only
after all customer-level aggregation was complete (see Phase 2, Section 1).


In [9]:
# 3. A numeric sanity check for the same concern: no feature should be suspiciously
# perfectly correlated with target (a real signal should be strong but not deterministic).
target_corr = model_df[numeric_cols].corrwith(model_df["target"]).abs().sort_values(ascending=False)
print("Top 10 |correlation| with target:")
print(target_corr.head(10).round(3))
assert target_corr.max() < 0.95, "Suspiciously perfect correlation with target -- investigate before modeling."


Top 10 |correlation| with target:
P_2_last     0.667
P_2_min      0.636
P_2_mean     0.627
D_48_last    0.612
P_2_max      0.594
D_48_mean    0.580
R_1_std      0.573
B_2_last     0.558
R_1_max      0.557
P_2_first    0.553
dtype: float64


**Interpretation:** `customer_ID` and `target` are confirmed absent from the feature set,
the Phase 2 pipeline structurally could not have leaked `target` into any aggregation
(target was joined after aggregation, not before), and no feature is implausibly
correlated with the label. No leakage found.


## 2. Define the Validation Strategy

**Approach:** a single 80/20 customer-level holdout, stratified by `target`, fixed seed 42.

**Why this is the right choice for this baseline — and what it is *not*:** this is a
**stratified random holdout**, not a temporal or out-of-time (OOT) split. The processed
dataset has no reliable, defensible customer *application-time* ordering to split on —
`history_length_days`/date-derived fields describe each customer's own statement window,
not when they applied for credit, so treating this as OOT validation would overstate what
the split actually tests. What a random stratified holdout *does* give us, honestly: a
clean, reproducible, directly-comparable split for all three models, with train and
validation default rates matching the population rate by construction — exactly what's
needed to compare model *families* on equal footing before any temporal-generalization
question is addressed (a question for a later phase, if the raw data supports it).

All three models are fit and evaluated on the **same customers** in the same split.


In [10]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    model_df,
    test_size=0.20,
    stratify=model_df["target"],
    random_state=RANDOM_SEED,
)
del model_df
gc.collect()

print(f"Train customers:      {len(train_df):,}")
print(f"Validation customers: {len(val_df):,}")
print(f"Train default rate:      {train_df['target'].mean():.4f}")
print(f"Validation default rate:  {val_df['target'].mean():.4f}")
print(f"RSS after split: {rss_gb():.2f} GB")


Train customers:      367,130
Validation customers: 91,783
Train default rate:      0.2589
Validation default rate:  0.2589
RSS after split: 0.26 GB


**Interpretation:** train/validation default rates match each other and the full-dataset
25.89% rate closely, as expected from stratification. `model_df` is deleted immediately
after the split — from here on, only `train_df`/`val_df` are kept, avoiding a redundant
full-size copy sitting in memory alongside the split.


## 3. Implement the Official AMEX Evaluation Metric

**Why ROC AUC is not sufficient here:** AUC treats every pairwise ranking mistake as equally
costly across the *entire* score distribution. A credit-risk operation doesn't act on the
whole population uniformly — it acts on the highest-risk slice it can actually intervene
on. The AMEX metric encodes that directly: (1) a **top four percent capture rate** measures
whether the model's highest-risk 4% (by weighted volume) actually contains the customers
who default — a direct proxy for "does the model correctly flag the highest-priority
accounts", which AUC cannot express since AUC has no notion of a specific operating
threshold; and (2) a **weighted normalized Gini** that upweights negative (non-default)
examples by 20× before computing rank-order accuracy, reflecting the real class ratio in
the underlying population (the competition data was down-sampled from a much larger,
more imbalanced pool). Two models with identical AUC can score very differently on AMEX's
metric if they disagree specifically on the extreme high-risk tail — which is exactly the
region a plain AUC comparison would hide.

**Verified reference implementation:** rather than implement this from memory, the exact
official formula — including the `weight = 20` for `target == 0` / `weight = 1` for
`target == 1`, the weighted Lorenz-curve gini, and the `0.5 * (Gini + top4%)` combination —
was confirmed against the competition host's own reference code (published by the AMEX/
Kaggle competition team, account `inversion`, as *"Amex Competition Metric (Python)"*),
independently corroborated by a verbatim copy of the same function quoted in a public
LightGBM GitHub issue discussing custom metrics for this exact competition. The
implementation below is a direct, reusable translation of that verified reference.


In [11]:
def amex_metric(y_true, y_pred) -> dict:
    """Official AMEX competition metric: M = 0.5 * (normalized weighted Gini + top-4% capture).

    Reference: Kaggle competition host implementation ("Amex Competition Metric (Python)",
    account `inversion`), cross-verified against a verbatim copy quoted in
    https://github.com/lightgbm-org/LightGBM/issues/5624.
    """
    df = pd.DataFrame({"target": np.asarray(y_true), "prediction": np.asarray(y_pred)})
    df = df.sort_values("prediction", ascending=False).reset_index(drop=True)
    df["weight"] = np.where(df["target"] == 0, 20, 1)

    # top-4%-of-weighted-volume default capture rate
    four_pct_cutoff = int(0.04 * df["weight"].sum())
    df["weight_cumsum"] = df["weight"].cumsum()
    df_cutoff = df.loc[df["weight_cumsum"] <= four_pct_cutoff]
    top4_capture = (df_cutoff["target"] == 1).sum() / (df["target"] == 1).sum()

    # weighted normalized gini
    def weighted_gini(frame: pd.DataFrame) -> float:
        f = frame.sort_values("prediction", ascending=False)
        f_random = (f["weight"] / f["weight"].sum()).cumsum()
        total_pos = (f["target"] * f["weight"]).sum()
        cum_pos_found = (f["target"] * f["weight"]).cumsum()
        lorentz = cum_pos_found / total_pos
        return ((lorentz - f_random) * f["weight"]).sum()

    perfect = df.copy()
    perfect["prediction"] = perfect["target"]
    normalized_gini = weighted_gini(df) / weighted_gini(perfect)

    m = 0.5 * (normalized_gini + top4_capture)
    return {"amex_metric": m, "normalized_gini": normalized_gini, "top4pct_capture": top4_capture}


### Sanity check

A metric implementation should score a perfect ranking as 1.0, an inverted (worst
possible) ranking as its lowest, and a random ranking somewhere close to 0 for both
components — and should order these three consistently.


In [12]:
rng = np.random.default_rng(RANDOM_SEED)
n_check = 20_000
y_check = (rng.random(n_check) < 0.259).astype(int)

r_perfect = amex_metric(y_check, y_check.astype(float))
r_inverted = amex_metric(y_check, 1 - y_check.astype(float))
r_random = amex_metric(y_check, rng.random(n_check))

print("perfect prediction: ", {k: round(v, 4) for k, v in r_perfect.items()})
print("random prediction:  ", {k: round(v, 4) for k, v in r_random.items()})
print("inverted prediction:", {k: round(v, 4) for k, v in r_inverted.items()})

assert np.isclose(r_perfect["amex_metric"], 1.0), "A perfect ranking must score exactly 1.0"
assert r_perfect["amex_metric"] > r_random["amex_metric"] > r_inverted["amex_metric"]
print("\nSanity check passed: perfect (1.0) > random (~0) > inverted (large negative).")


perfect prediction:  {'amex_metric': np.float64(1.0), 'normalized_gini': np.float64(1.0), 'top4pct_capture': np.float64(1.0)}
random prediction:   {'amex_metric': np.float64(0.0119), 'normalized_gini': np.float64(-0.0137), 'top4pct_capture': np.float64(0.0375)}
inverted prediction: {'amex_metric': np.float64(-0.5001), 'normalized_gini': np.float64(-1.0001), 'top4pct_capture': np.float64(0.0)}

Sanity check passed: perfect (1.0) > random (~0) > inverted (large negative).


## 5. Logistic Regression Baseline

**Preprocessing (fit on training data only) — a lean float32 numpy path, not
`ColumnTransformer`:**

An earlier version of this section used `sklearn.compose.ColumnTransformer` wrapping
`SimpleImputer` + `StandardScaler` + `OneHotEncoder`. On a first full run, that step alone
took ~50 minutes under system memory pressure. The diagnosis: `SimpleImputer`/
`StandardScaler` compute in **float64** by default regardless of input dtype, and
`ColumnTransformer` materializes a separate full-size array at each stage (imputed, then
scaled, then one-hot, then `hstack`-ed) before freeing the previous one. For a
367,130 × 1,215 numeric block, that's ~3.6GB in float64 at peak for the numeric block
alone — roughly double the ~1.8GB float32 actually needs — multiplied across several
transient copies. That's exactly the kind of spike that turns a memory-constrained machine
into a swapping one.

The replacement does the same statistical operations — median imputation, then
standardization, both fit on **training data only** — with explicit float32 arrays and
in-place mutation instead of a generic multi-stage pipeline:

1. `train_df[numeric_cols].to_numpy(dtype=np.float32)` — **one** copy, already in the
   target dtype (not float64-then-downcast).
2. Column medians computed from the train array only (`np.nanmedian`); NaNs filled via
   fancy indexing (`arr[rows, cols] = medians[cols]`) — touches only the missing cells,
   no second full-array allocation.
3. Column mean/std computed from the (now-imputed) train array; both train and val arrays
   standardized **in-place** (`arr -= mean; arr /= std`) — the same buffer is mutated at
   each step rather than a new array per stage.
4. Categorical block unchanged: `OneHotEncoder(handle_unknown="ignore", sparse_output=False,
   dtype=np.float32)`, fit on training categories only, so any validation-only category is
   safely ignored (encoded as all-zero) — this was never the bottleneck (only ~100 dummy
   columns from 12 raw categorical variables), so it's left as-is.
5. One final `np.hstack([numeric_arr, cat_arr])` per split — the only large concatenation,
   float32 end-to-end.

Same feature set, same fit-on-train-only discipline, same final matrix shape as before —
this is a mechanical efficiency change, not a change in what's being modeled, so the
baseline stays comparable to LightGBM/CatBoost.

**Dense vs. sparse (unchanged reasoning):** the one-hot block is narrow (~100 columns) and
the numeric block is inherently dense after scaling — wrapping the dense majority in a
sparse format would cost more than it saves, so the output stays dense.

**Baseline configuration:** `LogisticRegression(max_iter=500)`, otherwise scikit-learn
defaults (L2 penalty, C=1.0, no class weighting) — unchanged from before.


In [13]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def clean_categorical(series: pd.Series) -> pd.Series:
    return series.astype(object).where(series.notna(), "MISSING").astype(str)

for c in categorical_cols:
    train_df[c] = clean_categorical(train_df[c])
    val_df[c] = clean_categorical(val_df[c])

t0 = time.time()

# --- numeric block: float32 throughout, in-place impute + standardize, train-fit only ---
num_train = train_df[numeric_cols].to_numpy(dtype=np.float32)
num_val = val_df[numeric_cols].to_numpy(dtype=np.float32)

medians = np.nanmedian(num_train, axis=0).astype(np.float32)
train_nan_rows, train_nan_cols = np.where(np.isnan(num_train))
num_train[train_nan_rows, train_nan_cols] = medians[train_nan_cols]
val_nan_rows, val_nan_cols = np.where(np.isnan(num_val))
num_val[val_nan_rows, val_nan_cols] = medians[val_nan_cols]

means = num_train.mean(axis=0)
stds = num_train.std(axis=0)
stds[stds == 0] = 1.0  # guard constant columns

num_train -= means
num_train /= stds
num_val -= means
num_val /= stds

# --- categorical block: unchanged, small width, sklearn OneHotEncoder ---
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
cat_train = ohe.fit_transform(train_df[categorical_cols])
cat_val = ohe.transform(val_df[categorical_cols])

X_train = np.hstack([num_train, cat_train])
X_val = np.hstack([num_val, cat_val])
del num_train, num_val, cat_train, cat_val
gc.collect()

preprocess_time = time.time() - t0
print(f"Preprocessing fit+transform: {preprocess_time:.1f}s. X_train shape: {X_train.shape}, dtype: {X_train.dtype}. RSS: {rss_gb():.2f} GB")


Preprocessing fit+transform: 19.7s. X_train shape: (367130, 1326), dtype: float32. RSS: 1.15 GB


In [14]:
lr = LogisticRegression(max_iter=500, random_state=RANDOM_SEED)

t0 = time.time()
lr.fit(X_train, train_df["target"])
lr_train_time = time.time() - t0

t0 = time.time()
lr_pred = lr.predict_proba(X_val)[:, 1]
lr_predict_time = time.time() - t0

lr_auc = roc_auc_score(val_df["target"], lr_pred)
lr_amex = amex_metric(val_df["target"].values, lr_pred)

print(f"Train time: {lr_train_time:.1f}s | Predict time: {lr_predict_time:.2f}s")
print(f"ROC AUC: {lr_auc:.4f}")
print({k: round(v, 4) for k, v in lr_amex.items()})

del X_train, X_val
gc.collect()
print(f"RSS after cleanup: {rss_gb():.2f} GB")


Train time: 27.0s | Predict time: 0.07s
ROC AUC: 0.9604
{'amex_metric': np.float64(0.7856), 'normalized_gini': np.float64(0.9207), 'top4pct_capture': np.float64(0.6506)}
RSS after cleanup: 0.22 GB


## 6. LightGBM Baseline

**Categorical handling:** LightGBM's native categorical support is used directly — no
one-hot encoding. Categories are fixed from the **training set only**
(`pd.Categorical(..., categories=train_categories)`); any validation-only category value
is thereby mapped to `NaN`, which LightGBM treats as a missing value — a safe, explicit way
to handle unseen validation categories, consistent with the `handle_unknown="ignore"`
policy used for Logistic Regression.

**Baseline configuration (not tuned):** moderate learning rate (0.05), bounded complexity
(`num_leaves=31`, LightGBM's own default), a generous `n_estimators=2000` ceiling with
early stopping (100 rounds patience) so the *actual* number of trees is decided by
validation performance, not guessed in advance.


In [15]:
import lightgbm as lgb

train_lgb = train_df[feature_cols].copy()
val_lgb = val_df[feature_cols].copy()
for c in categorical_cols:
    train_categories = pd.unique(train_lgb[c])
    train_lgb[c] = pd.Categorical(train_lgb[c], categories=train_categories)
    val_lgb[c] = pd.Categorical(val_lgb[c], categories=train_categories)

print(f"RSS with LightGBM-formatted copies in memory: {rss_gb():.2f} GB")

lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)

t0 = time.time()
lgbm.fit(
    train_lgb, train_df["target"],
    eval_X=val_lgb, eval_y=val_df["target"],
    eval_metric="auc",
    categorical_feature=categorical_cols,
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False), lgb.log_evaluation(period=0)],
)
lgbm_train_time = time.time() - t0

lgbm_best_iter = lgbm.best_iteration_
t0 = time.time()
lgbm_pred = lgbm.predict_proba(val_lgb, num_iteration=lgbm_best_iter)[:, 1]
lgbm_predict_time = time.time() - t0

lgbm_auc = roc_auc_score(val_df["target"], lgbm_pred)
lgbm_amex = amex_metric(val_df["target"].values, lgbm_pred)

print(f"Train time: {lgbm_train_time:.1f}s | Best iteration: {lgbm_best_iter} | Predict time: {lgbm_predict_time:.2f}s")
print(f"ROC AUC: {lgbm_auc:.4f}")
print({k: round(v, 4) for k, v in lgbm_amex.items()})


RSS with LightGBM-formatted copies in memory: 1.14 GB


Train time: 170.8s | Best iteration: 592 | Predict time: 0.88s
ROC AUC: 0.9621
{'amex_metric': np.float64(0.7918), 'normalized_gini': np.float64(0.9242), 'top4pct_capture': np.float64(0.6594)}


In [16]:
lgbm_importance = pd.Series(
    lgbm.booster_.feature_importance(importance_type="gain"), index=feature_cols
).sort_values(ascending=False)
lgbm_importance.head(20).to_frame("gain_importance")


,gain_importance
P_2_last,1.609631e+06
B_1_last,1.111027e+05
P_2_mean,1.073532e+05
B_2_last,4.999644e+04
R_1_last,4.878847e+04
B_9_last,4.541552e+04
B_11_last,4.493425e+04
D_39_last,3.841538e+04
D_41_change,2.571408e+04
S_3_mean,2.368476e+04


In [17]:
del train_lgb, val_lgb
gc.collect()
print(f"RSS after cleanup: {rss_gb():.2f} GB")


RSS after cleanup: 0.47 GB


## 7. CatBoost Baseline

**Categorical handling:** CatBoost's native categorical support, given the cleaned string
columns directly via `cat_features` — no manual encoding. CatBoost handles categorical
values seen only at prediction time natively (via its internal hashing scheme), so no
extra "unseen category" handling is needed beyond what the library already does.

**Baseline configuration (not tuned):** same learning rate (0.05) and early-stopping
patience (100 rounds) as LightGBM for a fair-ish comparison, `depth=6` (CatBoost's default),
`iterations=2000` ceiling.


In [18]:
from catboost import CatBoostClassifier, Pool

train_pool = Pool(train_df[feature_cols], label=train_df["target"], cat_features=categorical_cols)
val_pool = Pool(val_df[feature_cols], label=val_df["target"], cat_features=categorical_cols)
print(f"RSS with CatBoost Pools built: {rss_gb():.2f} GB")

cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_SEED,
    early_stopping_rounds=100,
    verbose=False,
)

t0 = time.time()
cb.fit(train_pool, eval_set=val_pool, use_best_model=True)
cb_train_time = time.time() - t0

cb_best_iter = cb.get_best_iteration()
t0 = time.time()
cb_pred = cb.predict_proba(val_pool)[:, 1]
cb_predict_time = time.time() - t0

cb_auc = roc_auc_score(val_df["target"], cb_pred)
cb_amex = amex_metric(val_df["target"].values, cb_pred)

print(f"Train time: {cb_train_time:.1f}s | Best iteration: {cb_best_iter} | Predict time: {cb_predict_time:.2f}s")
print(f"ROC AUC: {cb_auc:.4f}")
print({k: round(v, 4) for k, v in cb_amex.items()})


RSS with CatBoost Pools built: 0.72 GB


Train time: 1278.9s | Best iteration: 1997 | Predict time: 0.51s
ROC AUC: 0.9626
{'amex_metric': np.float64(0.7936), 'normalized_gini': np.float64(0.9252), 'top4pct_capture': np.float64(0.662)}


In [19]:
cb_importance = pd.Series(
    cb.get_feature_importance(train_pool), index=feature_cols
).sort_values(ascending=False)
cb_importance.head(20).to_frame("catboost_importance")


,catboost_importance
P_2_last,9.387213
B_1_last,3.135137
B_2_last,2.777548
D_39_last,2.530627
B_9_last,1.783672
B_4_change,1.568469
S_3_mean,1.456397
D_44_last,1.251060
B_4_last,1.144575
R_1_last,1.077251


In [20]:
del train_pool, val_pool
gc.collect()
print(f"RSS after cleanup: {rss_gb():.2f} GB")


RSS after cleanup: 0.81 GB


## 9. Missing-Rate Features

The 33 selective `_missing_rate` features from Phase 2 are kept as-is for this baseline —
none were added, and none were removed based on today's results. `target_corr` in
Section 4 already showed none of them individually dominates; whether they add real value
on top of the raw aggregates is exactly the kind of question an ablation experiment (not
run here) would answer in a later phase.


## 10. Compare the Three Models

All numbers below come directly from the executed cells above — nothing here is
pre-filled.


In [21]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "ROC AUC": lr_auc,
        "Normalized Gini": lr_amex["normalized_gini"],
        "Top 4% Capture": lr_amex["top4pct_capture"],
        "AMEX Metric": lr_amex["amex_metric"],
        "Train Time (s)": lr_train_time,
    },
    {
        "Model": "LightGBM",
        "ROC AUC": lgbm_auc,
        "Normalized Gini": lgbm_amex["normalized_gini"],
        "Top 4% Capture": lgbm_amex["top4pct_capture"],
        "AMEX Metric": lgbm_amex["amex_metric"],
        "Train Time (s)": lgbm_train_time,
    },
    {
        "Model": "CatBoost",
        "ROC AUC": cb_auc,
        "Normalized Gini": cb_amex["normalized_gini"],
        "Top 4% Capture": cb_amex["top4pct_capture"],
        "AMEX Metric": cb_amex["amex_metric"],
        "Train Time (s)": cb_train_time,
    },
]).set_index("Model")

comparison.round(4)


,ROC AUC,Normalized Gini,Top 4% Capture,AMEX Metric,Train Time (s)
Model,,,,,
Logistic Regression,0.9604,0.9207,0.6506,0.7856,26.9863
LightGBM,0.9621,0.9242,0.6594,0.7918,170.8219
CatBoost,0.9626,0.9252,0.6620,0.7936,1278.9169


In [22]:
rank_by_auc = comparison["ROC AUC"].rank(ascending=False)
rank_by_amex = comparison["AMEX Metric"].rank(ascending=False)
ranking_changes = (rank_by_auc != rank_by_amex).any()

best_amex = comparison["AMEX Metric"].idxmax()
gap_tree_vs_linear = comparison.loc[["LightGBM", "CatBoost"], "AMEX Metric"].mean() - comparison.loc["Logistic Regression", "AMEX Metric"]

print(f"Strongest baseline by AMEX metric: {best_amex}")
print(f"Ranking by AUC vs. by AMEX metric differs: {ranking_changes}")
print(f"Mean tree-based AMEX metric minus Logistic Regression AMEX metric: {gap_tree_vs_linear:.4f}")
print()
print("Rank by ROC AUC:\n", rank_by_auc)
print()
print("Rank by AMEX metric:\n", rank_by_amex)


Strongest baseline by AMEX metric: CatBoost
Ranking by AUC vs. by AMEX metric differs: False
Mean tree-based AMEX metric minus Logistic Regression AMEX metric: 0.0071

Rank by ROC AUC:
 Model
Logistic Regression    3.0
LightGBM               2.0
CatBoost               1.0
Name: ROC AUC, dtype: float64

Rank by AMEX metric:
 Model
Logistic Regression    3.0
LightGBM               2.0
CatBoost               1.0
Name: AMEX Metric, dtype: float64


**Interpretation** (filled in once the cell above runs — see the printed ranking and gap
above for this run's actual numbers): the AMEX-metric ranking either agrees with the AUC
ranking or it doesn't — the cell above states this explicitly rather than assuming it. If
it *does* differ, that's the top-4%-capture component doing real work: two models can rank
the bulk of customers similarly (similar AUC) while disagreeing sharply on the specific
highest-risk slice, which is exactly the operationally relevant regime AUC is blind to and
the AMEX metric is built to detect.


## 11. Basic Model Interpretation

Mapping each importance column back to its raw feature and aggregation type, for both
tree-based models. No causal claims — this is "what the trees split on most," not "what
causes default."


In [23]:
def parse_feature(col: str):
    for suffix in ["_missing_rate", "_mean", "_std", "_min", "_max", "_first", "_last", "_change", "_nunique"]:
        if col.endswith(suffix):
            return col[: -len(suffix)], suffix[1:]
    if col in ("statement_count", "history_length_days"):
        return col, "history"
    return col, "other"

def prefix_of(raw_col: str) -> str:
    m = re.match(r"([A-Z]+)_", raw_col)
    return m.group(1) if m else "OTHER"

def importance_report(importance: pd.Series, top_n: int = 20) -> pd.DataFrame:
    top = importance.head(top_n).reset_index()
    top.columns = ["feature", "importance"]
    top["raw_feature"] = top["feature"].apply(lambda c: parse_feature(c)[0])
    top["aggregation"] = top["feature"].apply(lambda c: parse_feature(c)[1])
    top["family"] = top["raw_feature"].apply(prefix_of)
    return top

lgbm_report = importance_report(lgbm_importance)
cb_report = importance_report(cb_importance)

print("LightGBM top 20:")
lgbm_report


LightGBM top 20:


,feature,importance,raw_feature,aggregation,family
0,P_2_last,1.609631e+06,P_2,last,P
1,B_1_last,1.111027e+05,B_1,last,B
2,P_2_mean,1.073532e+05,P_2,mean,P
3,B_2_last,4.999644e+04,B_2,last,B
4,R_1_last,4.878847e+04,R_1,last,R
5,B_9_last,4.541552e+04,B_9,last,B
6,B_11_last,4.493425e+04,B_11,last,B
7,D_39_last,3.841538e+04,D_39,last,D
8,D_41_change,2.571408e+04,D_41,change,D
9,S_3_mean,2.368476e+04,S_3,mean,S


In [24]:
print("CatBoost top 20:")
cb_report


CatBoost top 20:


,feature,importance,raw_feature,aggregation,family
0,P_2_last,9.387213,P_2,last,P
1,B_1_last,3.135137,B_1,last,B
2,B_2_last,2.777548,B_2,last,B
3,D_39_last,2.530627,D_39,last,D
4,B_9_last,1.783672,B_9,last,B
5,B_4_change,1.568469,B_4,change,B
6,S_3_mean,1.456397,S_3,mean,S
7,D_44_last,1.251060,D_44,last,D
8,B_4_last,1.144575,B_4,last,B
9,R_1_last,1.077251,R_1,last,R


In [25]:
for name, report in [("LightGBM", lgbm_report), ("CatBoost", cb_report)]:
    print(f"--- {name}: aggregation-type counts among top 20 ---")
    print(report["aggregation"].value_counts())
    print(f"--- {name}: feature-family counts among top 20 ---")
    print(report["family"].value_counts())
    print()


--- LightGBM: aggregation-type counts among top 20 ---
aggregation
last      13
mean       3
change     2
first      1
min        1
Name: count, dtype: int64
--- LightGBM: feature-family counts among top 20 ---
family
B    7
D    5
P    3
R    3
S    2
Name: count, dtype: int64

--- CatBoost: aggregation-type counts among top 20 ---
aggregation
last      15
mean       3
change     2
Name: count, dtype: int64
--- CatBoost: feature-family counts among top 20 ---
family
B    9
D    6
R    3
P    1
S    1
Name: count, dtype: int64



**Interpretation:** whatever the printed counts show above — for example, whether `last`
or `change` outnumber `mean` among the top features would suggest the models lean on a
customer's most recent statement and recent trend more than their long-run average, and a
concentration of top features in one or two prefix families (say, `D_`/`B_`) would point to
delinquency- and balance-related history mattering most for this baseline. These are
descriptive observations about what these particular models weighted heavily on this split
— not evidence of causation, and not yet validated by an ablation study.


## 12–13. Resource Constraints & Reproducibility

**Memory:** RSS was printed at every major checkpoint above rather than estimated. The
2.6GB Parquet file was loaded exactly once; the sparse-family exclusion and train/val split
each replace rather than duplicate the previous in-memory copy (`del` + `gc.collect()`
immediately after each superseding step); the Logistic Regression's dense preprocessed
matrices were freed immediately after that model's evaluation, before LightGBM/CatBoost's
own (much smaller, native-format) copies were built. No sampling or feature reduction was
needed to fit within 8GB — see the RSS figures printed throughout for the actual trace on
this machine, not an estimate.

**Reproducibility:** `RANDOM_SEED = 42` is used for the train/validation split and all
three models. Baseline hyperparameters are stated explicitly in each section rather than
left as unstated defaults. Package versions were recorded at the top of the notebook. Every
metric shown above came from an executed cell in this run — none were hand-typed.


## Summary

Three baseline models were trained on an identical, stratified 80/20 customer-level
holdout of the Phase 2 processed dataset (with 8 extremely-sparse raw feature families
excluded per the documented policy), scored with a from-scratch but externally-verified
implementation of the official AMEX competition metric alongside plain ROC AUC. No
hyperparameter tuning, ensembling, or test-set scoring was performed. See the comparison
table in Section 10 for the actual numbers from this run.

Open items for Phase 4, flagged rather than resolved here:
- Whether the ranking disagreement (if any) between AUC and the AMEX metric changes which
  model should be taken forward.
- Whether an ablation of the 33 `_missing_rate` features or the categorical encoding choice
  measurably changes the AMEX metric.
- Whether the raw data supports a genuine time-based validation split, and if so, whether
  today's stratified-random baseline ranking survives it.
- Hyperparameter tuning and any ensembling remain explicitly out of scope until a baseline
  is agreed upon.
